## Setup

In [1]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study

from libs import constants as Cs
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from scipy.spatial.distance import pdist


SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 6

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-20 18:15:16.332965: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-20 18:15:16.364463: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-20 18:15:17.087855: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF

## Add Novelty Diff

In [2]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="lunarlander",
                container="add_novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                fitness_weight=fit_w,
                decay=decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=15)
study = create_study(direction="maximize", sampler=sampler, study_name="diff_add_novelty_exp_fixed_last", storage="sqlite:///Data/optuna/lunarlander/add_novelty/diff_fixed.db", load_if_exists=True)
study.optimize(objective, n_trials=160, n_jobs=5)
print(study.best_trials)

[I 2026-06-20 18:15:18,518] A new study created in RDB with name: diff_add_novelty_exp_fixed_last


Weight is now 0.9000000000000001 - decay4.5
Weight is now 0.9000000000000001 - decay4.5Weight is now 0.7 - decay5.0Weight is now 0.9000000000000001 - decay4.5


Weight is now 0.7 - decay5.0
Weight is now 0.7 - decay5.0
Weight is now 0.6000000000000001 - decay0.5
Weight is now 0.6000000000000001 - decay3.5Weight is now 0.6000000000000001 - decay3.5

Weight is now 0.6000000000000001 - decay3.5
Weight is now 0.6000000000000001 - decay0.5
Weight is now 1.0 - decay4.5Weight is now 1.0 - decay4.5

Weight is now 1.0 - decay4.5
Weight is now 0.6000000000000001 - decay0.5
Weight is now 0.9000000000000001 - decay4.5
Weight is now 0.6667363986135462 - decay4.5
Weight is now 0.7 - decay5.0
Weight is now 0.6000000000000001 - decay3.5
Weight is now 0.7 - decay5.0
Weight is now 0.4751337398020691 - decay3.5
Weight is now 0.5015719174016524 - decay5.0
Weight is now 0.5015719174016524 - decay5.0
Weight is now 0.9000000000000001 - decay4.5
Weight is now 0.6667363986135462 - decay4.5
Weight is now 0.6000

[I 2026-06-20 18:54:25,235] Trial 4 finished with value: -90.1275143539958 and parameters: {'lambda': 40, 'mutation_rate': 0.33, 'cross_rate': 0.6000000000000001, 'start_fit_w': 0.7, 'decay': 5.0}. Best is trial 4 with value: -90.1275143539958.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWa

Weight is now 0.30000000000000004 - decay0.5
Weight is now 0.30000000000000004 - decay0.5
Weight is now 0.30000000000000004 - decay0.5


[I 2026-06-20 18:54:32,484] Trial 3 finished with value: -63.575349061157254 and parameters: {'lambda': 40, 'mutation_rate': 0.45, 'cross_rate': 0.4, 'start_fit_w': 0.9000000000000001, 'decay': 4.5}. Best is trial 3 with value: -63.575349061157254.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: Runti

...

Weight is now 0.30000000000000004 - decay4.5
Weight is now 0.30000000000000004 - decay4.5
Weight is now 0.30000000000000004 - decay4.5
Weight is now 0.049589666466475966 - decay4.5
5  	-381.734	5  	-46.9707	-712.513	206.607	0.476059	5  	0.935504	0.00260161 	0.286546
Weight is now 0.036736928475894576 - decay4.5
Weight is now 0.036736928475894576 - decay4.5
6  	-348.98 	6  	-103.596	-616.413	141.674	0.478915	6  	0.995002	-0.0901376	0.296362
Weight is now 0.027215385986823763 - decay4.5
Weight is now 0.30000000000000004 - decay4.5
Weight is now 0.2222454662045154 - decay4.5
Weight is now 0.30000000000000004 - decay4.5
Weight is now 0.2222454662045154 - decay4.5
Weight is now 0.30000000000000004 - decay4.5
Weight is now 0.2222454662045154 - decay4.5
Weight is now 0.049589666466475966 - decay4.5
5  	-460.891	5  	-87.9993	-855.155	194.457	0.329314	5  	0.861633	-0.246127	0.275156
Weight is now 0.036736928475894576 - decay4.5
Weight is now 0.036736928475894576 - decay4.5
6  	-394.603	6  	-100

[I 2026-06-21 14:01:31,829] Trial 158 finished with value: 84.49098901156889 and parameters: {'lambda': 40, 'mutation_rate': 0.18, 'cross_rate': 0.4, 'start_fit_w': 0.30000000000000004, 'decay': 4.5}. Best is trial 70 with value: 85.32851456392943.


Weight is now 0.003332698961472693 - decay4.5
14 	-464.948	14 	55.4754 	-692.417	196.943	0.299327	14 	0.979764	0.00124649	0.257352


[I 2026-06-21 14:04:28,382] Trial 159 finished with value: 84.49098901156889 and parameters: {'lambda': 40, 'mutation_rate': 0.18, 'cross_rate': 0.4, 'start_fit_w': 0.30000000000000004, 'decay': 4.5}. Best is trial 70 with value: 85.32851456392943.
Process ForkProcess-129:
Process ForkProcess-150:
Process ForkProcess-149:
Process ForkProcess-292:
Process ForkProcess-127:
Process ForkProcess-294:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
 

KeyboardInterrupt: 

## Sub Novelty

In [ ]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="lunarlander",
                container="sub_novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                fitness_weight=fit_w,
                decay=decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    behaviors = list(map(lambda x:x[1], fitnesses))
    fitnesses = list(map(lambda x:x[0], fitnesses))
    diversity = pdist(np.array(behaviors)).mean()
    assert diversity > 0
    return np.max(fitnesses) + diversity


sampler = TPESampler(n_startup_trials=15)
study = create_study(direction="maximize", sampler=sampler, study_name="diff_sub_novelty_exp_repair_last", storage="sqlite:///Data/optuna/lunarlander/sub_novelty/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=170, n_jobs=5)
print(study.best_trials)

[I 2026-06-21 19:36:09,289] Using an existing study with name 'diff_sub_novelty_exp_repair' instead of creating a new one.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessTrue' has already been created and it will be overwritten. Consider deleting previous creatio

KeyboardInterrupt: 

## Fit_Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="fit_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_fit_archiving_reloaded", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_fita_study.best_trials)

## Novlety Limit Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    limit = trial.suggest_float("limit", -200, -50, step=10)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="novelty_limit",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                limit=limit,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_novelty_limit", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty_limit/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=170, n_jobs=5)
print(diff_fita_study.best_trials)

## Novelty Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="novelty_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_novelty_archiving", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty_archiving/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_fita_study.best_trials)